This notebook was used to generate the UN General Debate corpus chunked dataset available at

https://huggingface.co/datasets/kalebr/un-general-debate-corpus-chunked

We begin by fetching the original dataset, which was prepared by Alexander Baturo, Niheer Dasandi, and Slava Mikhaylov, and is presented in the paper "Understanding State Preferences With Text As Data: Introducing the UN General Debate Corpus" Research & Politics, 2017.

In [2]:
!curl -L -o un-general-debates.zip\
  https://www.kaggle.com/api/v1/datasets/download/unitednations/un-general-debates

!unzip un-general-debates.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 43.6M    0 85960    0     0   222k      0  0:03:20 --:--:--  0:03:20  222k^C
Archive:  un-general-debates.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of un-general-debates.zip or
        un-general-debates.zip.zip, and cannot find un-general-debates.zip.ZIP, period.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('un-general-debates.csv')
df.head()

,session,year,country,text
0,44,1989,MDV,﻿It is indeed a pleasure for me and the member...
1,44,1989,FIN,"﻿\nMay I begin by congratulating you. Sir, on ..."
2,44,1989,NER,"﻿\nMr. President, it is a particular pleasure ..."
3,44,1989,URY,﻿\nDuring the debate at the fortieth session o...
4,44,1989,ZWE,﻿I should like at the outset to express my del...


In [1]:
import torch

if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
else:
    print("GPU not available, using CPU")
    device = torch.device("cpu")

GPU not available, using CPU


Here we use the Chonkie library's semantic chunker to chunk each speech.

In [2]:

from tqdm import tqdm
import pandas as pd
from chonkie import SemanticChunker, AutoEmbeddings
import numpy as np

BATCH_SIZE = 256  # Adjust depending on GPU memory

all_chunks = []

# Stage 1: Fast embedding for chunking
fast_embedder = AutoEmbeddings.get_embeddings("all-MiniLM-L6-v2")
chunker = SemanticChunker(embedding_model=fast_embedder, threshold=0.8, chunk_size=512)

# Stage 2: High-quality embeddings
hq_embedder = AutoEmbeddings.get_embeddings("all-mpnet-base-v2")

# Step 1: Chunk all documents first
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Chunking docs"):
    text = row['text']
    chunks = chunker.chunk(text)

    for chunk in chunks:
        # Store chunk text + metadata for batching later
        if chunk.token_count > 100:
            all_chunks.append({
                'original_index': idx,
                'original_metadata': row.to_dict(),
                'chunk_text': chunk.text,
                'token_count': chunk.token_count
            })




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Chunking docs:   0%|          | 13/7507 [00:44<7:04:46,  3.40s/it]


KeyboardInterrupt: 

In [ ]:
# Step 2: Batch embed all chunks
chunk_texts = [c['chunk_text'] for c in all_chunks]

embeddings = []
for i in tqdm(range(0, len(chunk_texts), BATCH_SIZE), desc="Computing embeddings"):
    batch_texts = chunk_texts[i:i+BATCH_SIZE]
    batch_embeddings = hq_embedder.embed(batch_texts)  # This should handle batches
    embeddings.extend(batch_embeddings)

# Step 3: Attach embeddings back to all_chunks
for i, emb in enumerate(embeddings):
    all_chunks[i]['embedding'] = np.array(emb, dtype=np.float32)

# Step 4: Convert to DataFrame
chunk_df = pd.DataFrame(all_chunks)

for key in chunk_df['original_metadata'].values[0].keys():
  chunk_df[key] = chunk_df['original_metadata'].apply(lambda x: x[key])
chunk_df = chunk_df.drop(columns=['original_metadata'])
chunk_df = chunk_df.drop(columns=['text'])
chunk_df.head()

print(chunk_df.head())

Computing embeddings: 100%|██████████| 217/217 [13:40<00:00,  3.78s/it]


   original_index                                  original_metadata  \
0               0  {'session': 44, 'year': 1989, 'country': 'MDV'...   
1               0  {'session': 44, 'year': 1989, 'country': 'MDV'...   
2               0  {'session': 44, 'year': 1989, 'country': 'MDV'...   
3               0  {'session': 44, 'year': 1989, 'country': 'MDV'...   
4               0  {'session': 44, 'year': 1989, 'country': 'MDV'...   

                                          chunk_text  token_count  \
0  ﻿It is indeed a pleasure for me and the member...          107   
1  As in previous years, my delegation wishes to ...          132   
2  Recent years have witnessed a welcome positive...          173   
3  This, therefore, fuels the arms race and defen...          161   
4  The confidence that can he derived from genuin...          184   

                                           embedding  
0  [0.0061351038, 0.020647585, 0.01789057, 0.0222...  
1  [0.04616816, 0.05086965, 0.029461894, -

In [ ]:
from umap import UMAP

reducer = UMAP(verbose=True)
reduced_vectors = reducer.fit_transform(np.stack(chunk_df['embedding'].values))
chunk_df['reduced'] = reduced_vectors.tolist()

UMAP( verbose=True)
Sat Mar 14 18:44:53 2026 Construct fuzzy simplicial set
Sat Mar 14 18:44:53 2026 Finding Nearest Neighbors
Sat Mar 14 18:44:54 2026 Building RP forest with 17 trees
Sat Mar 14 18:45:04 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	Stopping threshold met -- exiting after 4 iterations
Sat Mar 14 18:45:43 2026 Finished Nearest Neighbor Search
Sat Mar 14 18:45:48 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Sat Mar 14 18:46:22 2026 Finished embedding


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from vectorizers.transformers  import InformationWeightTransformer

# Step 1: Convert text to term frequency counts
count_vectorizer = CountVectorizer(max_features=1000)
count_matrix = count_vectorizer.fit_transform(chunk_df['chunk_text'],y=chunk_df['original_index'])

# Step 2: Apply TF-IDF transformation to the count matrix
#fidf_transformer = TfidfTransformer()
iw_transformer = InformationWeightTransformer()
iw_matrix = iw_transformer.fit_transform(count_matrix)

# Sum TF-IDF scores per chunk (informativeness score)
chunk_df['information_weight'] = iw_matrix.sum(axis=1).A1

chunk_df.head()

,original_index,chunk_text,token_count,embedding,reduced,session,year,country,text,information_weight
0,0,﻿It is indeed a pleasure for me and the member...,107,"[0.0061351038, 0.020647585, 0.01789057, 0.0222...","[6.317007541656494, 7.351022243499756]",44,1989,MDV,﻿It is indeed a pleasure for me and the member...,39.726044
1,0,"As in previous years, my delegation wishes to ...",132,"[0.04616816, 0.05086965, 0.029461894, -0.02130...","[6.787918567657471, 4.686098098754883]",44,1989,MDV,﻿It is indeed a pleasure for me and the member...,30.712829
2,0,Recent years have witnessed a welcome positive...,173,"[0.05059605, 0.05956089, 0.012747274, -0.02632...","[1.7494146823883057, 8.707806587219238]",44,1989,MDV,﻿It is indeed a pleasure for me and the member...,47.679643
3,0,"This, therefore, fuels the arms race and defen...",161,"[0.01842546, 0.070237, 0.0534881, -0.024107208...","[8.963186264038086, -0.49657195806503296]",44,1989,MDV,﻿It is indeed a pleasure for me and the member...,41.543051
4,0,The confidence that can he derived from genuin...,184,"[0.054252107, 0.05724938, 0.016085196, -0.0274...","[9.401809692382812, -1.3375420570373535]",44,1989,MDV,﻿It is indeed a pleasure for me and the member...,62.701161


In [ ]:
chunk_df.to_parquet("ungdc-all-chunked.parquet")